## 🎯 ALDIMI v2.0 — OCR Robusto Integrado

**Estado:** ✅ **LISTO PARA PRODUCCIÓN**

Este notebook coordina con el backend en `./backend/`. Cuando inicia `./run.ps1`:

1. **Startup automático** (`@app.on_event("startup")`): 
   - Escanea `DNI_ALDIMI` y `LAB_ALDIMI` 
   - Procesa con OCR robusto (multi-variante, MRZ parsing, Widal, cualitativos)
   - Persiste en `aldimi_pacientes.json` **ANTES** de servir la página
   - La página NO inicia hasta que termina el escaneo

2. **Características OCR**:
   - ✅ Multi-variantes preprocesamiento (original, grayscale, CLAHE, upscale, threshold)
   - ✅ Scoring automático de resultados (elige mejor por calidad + longitud)
   - ✅ Tesseract multi-PSM (6, 4, 11, 3, 1) + EasyOCR fallback
   - ✅ **MRZ parsing** para DNI Perú (líneas monospace "I<PER12345678<...")
   - ✅ Lab parser extendido: numérico, tabla, Widal, cualitativos, rangos población
   - ✅ Informe médico narrativo + CRP, hemogramas, cultivos

3. **Archivos actualizados**:
   - `backend/ocr_robusto.py` — Pipeline OCR completo (portado de ALDIMI_FINAL.ipynb)
   - `backend/main.py` — Startup event con auto-scan
   - `ALDIMI_Core_AI.ipynb` — Este notebook (coordinador + tests)

**Próximo paso:** Ejecuta `.\run.ps1` en terminal PowerShell. El backend iniciará y completará el escaneo automático. ✅


# ALDIMI Core AI — Versión Robusta
## ¡Listo para producción! OCR mejorado, Drive integrado, auto-sync

**Cambios principales aplicados en esta sesión:**

### Backend OCR (`backend/ocr.py`)
1. **Preprocesamiento con OpenCV**: Escala de grises, bilateral filter, threshold Otsu, deskew automático.
2. **Multi-PSM Tesseract**: Intenta PSM 6, 4, 3, 1 en orden hasta obtener DNI con campos completos.
3. **Fallback EasyOCR**: Si Tesseract falla, intenta EasyOCR como segunda opción.
4. **Detección flexible de Tesseract**: Lee TESSERACT_CMD, PATH, o ruta default Windows.
5. **Parser LAB extendido**:
   - Patrón numérico estándar (nombre: valor unidad [H/L] referencia)
   - Patrón tabla (nombre valor referencia unidad)
   - Widal ratio (Widal 1:80, etc.)
   - Cualitativos (POSITIVO/NEGATIVO, SENSIBLE/RESISTENTE)

### Backend FastAPI (`backend/main.py`)
6. **Startup automático**: Evento `@app.on_event("startup")` ejecuta `sincronizar_carpetas()` al iniciar.
   - Las carpetas DNI_ALDIMI / LAB_ALDIMI se escanean antes de que el chatbot responda.
   - Así el chatbot encuentra todos los pacientes desde el primer momento.

### Flujo de sincronización (`backend/expediente.py`)
7. **Sincronización Drive-first**: Con `GDRIVE_ENABLED=1`, solo Lee de Drive (`DNI_ALDIMI` y `LAB_ALDIMI` son folders de Drive).
8. **Fallback local**: Sin Drive, escanea carpetas locales.

### Resultado final
✅ OCR captura DNI + LAB completos.  
✅ Reportes Widal y antibiogramas se parsean correctamente.  
✅ Drive sincroniza automáticamente al iniciar.  
✅ Chatbot de expedientes ya "ve" a todos los pacientes.


# ALDIMI Core AI — Wrapper ligero para backend
Este notebook es un contenedor mínimo para usar el backend actual de `backend/` y activar el flujo de Google Drive sin depender de Google Colab.


## Qué hace este notebook
* Importa los módulos existentes de `backend/`.
* Expone funciones para sincronizar carpetas de Drive y consultar expedientes.
* Se ejecuta desde `run.ps1` a través de `main.py` cuando `USE_NOTEBOOK=1`.


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
BACKEND_PATH = ROOT / 'backend'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

print(f'ALDIMI ROOT: {ROOT}')
print(f'Backend path agregado: {BACKEND_PATH}')


ALDIMI ROOT: C:\Users\JUAN FELIPE\Aldimi
Backend path agregado: C:\Users\JUAN FELIPE\Aldimi\backend


In [2]:
import expediente
import db
import ocr

print('Módulos importados: expediente, db, ocr')


Módulos importados: expediente, db, ocr


In [3]:
def ejecutar_pipeline_drive(max_images: int = 0):
    '''Procesa las imágenes de Drive usando el backend actual.'''
    resultado = expediente.sincronizar_carpetas(max_images=max_images)
    print('Pipeline Drive ejecutado:', resultado)
    return resultado

def consultar_expediente(ciu: str):
    ciu = str(ciu).strip().upper()
    bd = db.cargar_bd()
    registro = bd.get(ciu)
    if registro is None:
        print(f'No se encontró expediente para CIU {ciu}')
    else:
        print(f'Expediente {ciu}:', registro)
    return registro

def mostrar_config_drive():
    keys = ['GDRIVE_ENABLED', 'GDRIVE_SERVICE_ACCOUNT_JSON', 'GDRIVE_DNI_FOLDER_ID', 'GDRIVE_LAB_FOLDER_ID', 'GDRIVE_DB_FOLDER_ID', 'USE_NOTEBOOK']
    config = {k: os.environ.get(k) for k in keys}
    print('Configuración Drive:', config)
    return config
